# Hello, answer — a grounded answer with citations

`ask()` returns chunks. `answer()` takes those chunks, hands them to a local Ollama chat
model, and returns text that cites them by number. Citations pointing past the context are
stripped, so a `[7]` when only four chunks were retrieved never reaches you.

**Needs:** Ollama with `nomic-embed-text` *and* a chat model — `ollama pull llama3.2:3b`.

**This is the slow notebook.** On a loaded CPU box one answer took 127 seconds. The
retrieval underneath it took milliseconds; the wait is entirely the local model.

The `add` below reports `0 embedded, 3 unchanged` if you have already run
`00_hello_topic` — same store, same text, so nothing is re-embedded.

In [1]:
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

from slim_llm_memory import topic

t = topic("hello", path=ROOT / ".hello_nb" / "topic")
t.add({
    "nginx.md": "To serve a site with nginx, install the package, put a server block in "
                "/etc/nginx/sites-enabled, then run `nginx -s reload` to pick it up.",
    "carbonara.md": "Carbonara is guanciale, pecorino romano, egg yolks and black pepper. "
                    "No cream, and the pan comes off the heat before the egg goes in.",
    "backups.md": "Backups run nightly at 03:00, are encrypted with age, and are kept for "
                  "30 days before rotation deletes them.",
})

added 3 doc(s), 3 chunks: 0 embedded, 3 unchanged, 0 removed

In [2]:
a = t.answer("how do I reload the web server config?", model="llama3.2:3b")
print(a)

To reload the web server config with nginx, run `nginx -s reload` [1].


`a` is a plain `str`, so it prints and concatenates like one. What it carries besides
the text is the audit trail: which chunks were retrieved, which of them the model actually
cited, and whether it refused.

In [3]:
a.citations, a.refused, len(a.hits)

([1], False, 3)

## Refusing instead of inventing

`refuse_below` skips the model entirely when the best hit is too weak. Nothing in this
store is about tax law, so this question is refused without an LLM call — and without a
confident wrong answer.

In [4]:
b = t.answer("what is the capital gains rate in Portugal?",
             model="llama3.2:3b", refuse_below=0.55)
print(b)
print("refused:", b.refused)

I don't have anything about that in this store.
refused: True


In [5]:
t.close()